# Notebook: Qualitative Evaluation (Expert Analysis) -- Experiment 2

In this notebook, the qualitative evaluation of Experiment 2 is conducted for the question-generation behind RQ2 and RQ3.

The notebook uses one evaluation setup with exactly 3 experts. \
Question and answer quality criteria are rated on a 1-5 Likert scale and analyzed separately and as combined total scores.  
The Bloom level itself is rated on a 1-6 scale by experts, and Bloom alignment is derived as alignment points (1/3/5).

Question and answer ratings are analyzed separately and as combined total scores for the non-Bloom Likert criteria. Bloom alignment is analyzed separately using its own 1-6 scale.

## Initial Setup

In [ ]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
import warnings
from statsmodels.stats.inter_rater import fleiss_kappa
import pingouin as pg

warnings.filterwarnings('ignore')

plt.style.use('default')
sns.set_palette('Set2')
plt.rcParams['figure.dpi'] = 300
plt.rcParams['savefig.dpi'] = 300

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', 100)
pd.set_option('display.notebook_repr_html', True)

LIKERT_MIN = 1
LIKERT_MAX = 5
LIKERT_RANGE = (LIKERT_MIN, LIKERT_MAX)
BLOOM_MIN = 1
BLOOM_MAX = 6
BLOOM_RANGE = (BLOOM_MIN, BLOOM_MAX)
ALIGNMENT_POINT_RANGE = (1, 5)
EXPERT_IDS = [1, 2, 3]
ANALYSIS_SUFFIX = 'experts'

EXP2_QUESTION_COLS = [
    'q_clarity',
    'q_challenging',
    'q_value',
    'q_language',
]
EXP2_ANSWER_COLS = [
    'a_clarity',
    'a_language',
]
EXP2_BLOOM_COLS = [
    'q_bloom_rating',
    'a_bloom_rating',
]
EXP2_ALIGNMENT_COLS = [
    'q_bloom_alignment',
    'a_bloom_alignment',
    'bloom_alignment_total',
]
EXP2_NUMERIC_COLS = EXP2_QUESTION_COLS + EXP2_ANSWER_COLS + EXP2_ALIGNMENT_COLS
EXP2_SCORE_COLS = ['question_total_score', 'answer_total_score', 'total_score']

SCORE_RANGES = {
    'question_total_score': (5, 25),
    'answer_total_score': (3, 15),
    'total_score': (8, 40),
    'q_bloom_alignment': ALIGNMENT_POINT_RANGE,
    'a_bloom_alignment': ALIGNMENT_POINT_RANGE,
    'bloom_alignment_total': (2, 10),
    'question_total_with_alignment': (5, 25),
    'answer_total_with_alignment': (3, 15),
    'total_with_alignment': (8, 40),
}

DISPLAY_LABELS = {
    'q_clarity': 'qClarity',
    'q_challenging': 'qChallenging',
    'q_value': 'qValue',
    'q_language': 'qLanguage',
    'q_bloom_rating': 'qBloomRating',
    'a_clarity': 'aClarity',
    'a_language': 'aLanguage',
    'a_bloom_rating': 'aBloomRating',
    'question_total_score': 'qTotal',
    'answer_total_score': 'aTotal',
    'total_score': 'total',
    'q_bloom_alignment': 'qBloomAlignment',
    'a_bloom_alignment': 'aBloomAlignment',
    'bloom_alignment_total': 'bloomAlignmentTotal',
    'question_total_with_alignment': 'qTotalWithAlignment',
    'answer_total_with_alignment': 'aTotalWithAlignment',
    'total_with_alignment': 'totalWithAlignment',
}

BASE_PROJECT_PATH = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
qualitative_base_path = os.path.join(BASE_PROJECT_PATH, '20_experiments/60_analyses/csv/qualitative/exp2')
sampled_hints_path = os.path.join(BASE_PROJECT_PATH, '20_experiments/60_analyses/csv/sampled/exp2_sampled.csv')
output_base_path = os.path.join(BASE_PROJECT_PATH, '40_evaluation/exp2/qualitative')
output_tables_path = os.path.join(output_base_path, 'tables')
output_plots_path = os.path.join(output_base_path, 'plots')

for path in [output_tables_path, output_plots_path]:
    os.makedirs(path, exist_ok=True)

LABEL_MAPPING = {
    'anthropic': 'Anthropic',
    'openai': 'OpenAI',
    'deepseek': 'DeepSeek',
    'xai': 'xAI',
    'google': 'Google',
    'mcq': 'MCQ',
    'open_ended': 'Open-Ended',
}

tables = {}
plots = {}

print('Setup completed successfully')
print(f'Output tables: {output_tables_path}')
print(f'Output plots: {output_plots_path}')
print(f'Experts configured: {EXPERT_IDS}')
print('Likert scale: 1-5 | Bloom source scale: 1-6 | Alignment points: 1/3/5')

In [ ]:
# Utility Functions

def normalize_sample_id(series):
    return series.astype(str).str.extract(r'(\d+)')[0].str.zfill(3)


def create_seaborn_boxplot(data, x, y, ax, title, ylabel, xlabel, scale_range=None):
    plot_df = data[[x, y]].dropna().copy()
    if plot_df.empty:
        ax.set_title(f'{title} - No data', fontsize=12, fontweight='bold')
        ax.text(0.5, 0.5, 'No data available', ha='center', va='center', transform=ax.transAxes)
        ax.set_axis_off()
        return

    colors = ['#66c2a5', '#fc8d62', '#8da0cb', '#e78ac3', '#a6d854', '#ffd92f', '#e5c494', '#b3b3b3']
    order = sorted(plot_df[x].dropna().unique(), key=lambda value: str(value))
    palette = colors[:len(order)]

    sns.boxplot(
        data=plot_df,
        x=x,
        y=y,
        order=order,
        ax=ax,
        palette=palette,
        medianprops={'color': 'black', 'linewidth': 2.5, 'linestyle': ':'},
    )

    for i, value in enumerate(order):
        mean_val = plot_df.loc[plot_df[x] == value, y].mean()
        ax.scatter(i, mean_val, color='red', marker='D', s=50, zorder=3, edgecolor='darkred', linewidth=1)

    ax.set_xticklabels([LABEL_MAPPING.get(str(label), str(label)) for label in order], rotation=0)

    if scale_range is not None:
        padding = 0.1 if scale_range[1] <= 6 else 0.5
        ax.set_ylim(scale_range[0] - padding, scale_range[1] + padding)

    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_ylabel(ylabel, fontsize=11, labelpad=15)
    ax.set_xlabel(xlabel, fontsize=11, labelpad=10)
    ax.grid(True, alpha=0.3)


def create_heatmap(data, index_col, column_col, value_col, ax, title, cbar_label, scale_range=None):
    heatmap_mean = data.groupby([index_col, column_col])[value_col].mean().unstack()
    heatmap_std = data.groupby([index_col, column_col])[value_col].std().unstack()

    annot_matrix = heatmap_mean.copy().astype(object)
    for i in range(len(heatmap_mean.index)):
        for j in range(len(heatmap_mean.columns)):
            mean_val = heatmap_mean.iloc[i, j]
            std_val = heatmap_std.iloc[i, j]
            if pd.notna(mean_val) and pd.notna(std_val):
                annot_matrix.iloc[i, j] = f'{mean_val:.2f}\n({std_val:.2f})'
            elif pd.notna(mean_val):
                annot_matrix.iloc[i, j] = f'{mean_val:.2f}'
            else:
                annot_matrix.iloc[i, j] = ''

    yticklabels = [LABEL_MAPPING.get(str(label), str(label).title()) for label in heatmap_mean.index]
    xticklabels = [LABEL_MAPPING.get(str(label), str(label).title()) for label in heatmap_mean.columns]

    heatmap_kwargs = {
        'annot': annot_matrix,
        'fmt': '',
        'cmap': 'RdYlBu_r',
        'square': True,
        'linewidths': 0.5,
        'cbar_kws': {'shrink': 0.8, 'label': cbar_label},
        'annot_kws': {'size': 10, 'weight': 'bold'},
        'xticklabels': xticklabels,
        'yticklabels': yticklabels,
        'ax': ax,
    }

    if scale_range is not None:
        heatmap_kwargs['vmin'] = scale_range[0]
        heatmap_kwargs['vmax'] = scale_range[1]
        heatmap_kwargs['center'] = (scale_range[0] + scale_range[1]) / 2
    elif not heatmap_mean.empty:
        heatmap_kwargs['center'] = heatmap_mean.mean().mean()

    sns.heatmap(heatmap_mean, **heatmap_kwargs)
    ax.set_title(title, fontsize=14, fontweight='bold', pad=20)
    ax.set_xlabel(LABEL_MAPPING.get(column_col, column_col.replace('_', ' ').title()), fontsize=12, fontweight='bold')
    ax.set_ylabel(LABEL_MAPPING.get(index_col, index_col.replace('_', ' ').title()), fontsize=12, fontweight='bold')


print('Utility functions loaded successfully')

In [ ]:
def enrich_exp2_metadata(df):
    if not os.path.exists(sampled_hints_path):
        raise FileNotFoundError(f'Sampled file not found at {sampled_hints_path}')

    sampled_df = pd.read_csv(sampled_hints_path).copy()
    sampled_df['sample_id'] = [f'{i + 1:03d}' for i in range(len(sampled_df))]

    df = df.copy()
    df['sample_id'] = normalize_sample_id(df['sample_id'])

    exp2_df = pd.merge(
        df,
        sampled_df[['sample_id', 'llm', 'question_type', 'bloom_idx']],
        on='sample_id',
        how='left',
        suffixes=('', '_meta'),
    )

    if 'question_type_meta' in exp2_df.columns:
        if 'question_type' in exp2_df.columns:
            exp2_df['question_type'] = exp2_df['question_type'].combine_first(exp2_df['question_type_meta'])
        else:
            exp2_df['question_type'] = exp2_df['question_type_meta']
        exp2_df = exp2_df.drop(columns=['question_type_meta'])

    if 'question_type' in exp2_df.columns:
        valid_types = ['mcq', 'open_ended']
        exp2_df['question_type'] = exp2_df['question_type'].where(exp2_df['question_type'].isin(valid_types), pd.NA)

    exp2_df['bloom_idx'] = pd.to_numeric(exp2_df.get('bloom_idx'), errors='coerce')

    return exp2_df


def load_all_experts_data():
    experts_data = {}
    all_expert_dfs = []

    for expert_num in EXPERT_IDS:
        expert_key = f'expert_{expert_num}'
        expert_file = os.path.join(qualitative_base_path, f'exp2_eval_e{expert_num}.csv')

        if not os.path.exists(expert_file):
            print(f'Skip {expert_key}: file missing ({expert_file})')
            continue

        expert_df = pd.read_csv(expert_file)

        if expert_df.empty:
            print(f'Skip {expert_key}: empty CSV')
            continue

        first_row = expert_df.iloc[0]
        empty_count = first_row.apply(
            lambda v: pd.isna(v) or (isinstance(v, str) and v.strip() == '')
        ).sum()

        if empty_count >= 4:
            print(f'Skip {expert_key}: first data row has {empty_count} empty fields (>=4)')
            continue

        expert_df = enrich_exp2_metadata(expert_df)
        expert_df['expert'] = expert_key
        experts_data[expert_key] = expert_df.copy()
        all_expert_dfs.append(expert_df)

    if not all_expert_dfs:
        raise ValueError('No usable expert CSV found.')

    exp2_df = pd.concat(all_expert_dfs, ignore_index=True)
    return exp2_df, experts_data

In [ ]:
def clean_numeric_data(df, numeric_cols, min_value=1, max_value=5, use_semicolon_max=False):
    invalid_values = [
        '??', '???', '?', '????', '', ' ', 'nan', 'NaN', 'NULL', 'null',
        'None', 'NONE', 'n/a', 'N/A', '#N/A', '#NULL!',
        'undefined', 'UNDEFINED', '-', '--', '---', '-99',
    ]

    def _parse_value(value):
        if pd.isna(value):
            return np.nan

        value_str = str(value).strip()
        if value_str in invalid_values:
            return np.nan

        if use_semicolon_max and ';' in value_str:
            parts = [part.strip() for part in value_str.split(';') if part.strip() != '']
            numeric_parts = pd.to_numeric(pd.Series(parts), errors='coerce').dropna()
            if numeric_parts.empty:
                return np.nan
            return float(numeric_parts.max())

        return pd.to_numeric(value_str, errors='coerce')

    df = df.copy()
    for col in numeric_cols:
        if col in df.columns:
            df[col] = df[col].apply(_parse_value)
            df.loc[(df[col] < min_value) | (df[col] > max_value), col] = np.nan
    return df


def bloom_alignment_points(actual_rating, target_rating):
    if pd.isna(actual_rating) or pd.isna(target_rating):
        return np.nan

    actual = int(actual_rating)
    target = int(target_rating)
    diff = abs(actual - target)

    if diff == 0:
        return 5
    if diff == 1:
        return 3
    return 1


def calculate_exp2_scores(df):
    df = df.copy()

    df['q_bloom_alignment'] = df.apply(
        lambda row: bloom_alignment_points(row.get('q_bloom_rating'), row.get('bloom_idx')),
        axis=1,
    )
    df['a_bloom_alignment'] = df.apply(
        lambda row: bloom_alignment_points(row.get('a_bloom_rating'), row.get('q_bloom_rating')),
        axis=1,
    )
    df['bloom_alignment_total'] = df[['q_bloom_alignment', 'a_bloom_alignment']].sum(
        axis=1, min_count=2
    )

    q_cols = EXP2_QUESTION_COLS + ['q_bloom_alignment']
    a_cols = EXP2_ANSWER_COLS + ['a_bloom_alignment']

    df['question_total_score'] = df[q_cols].sum(axis=1, min_count=len(q_cols))
    df['answer_total_score'] = df[a_cols].sum(axis=1, min_count=len(a_cols))
    df['total_score'] = df[q_cols + a_cols].sum(axis=1, min_count=len(q_cols + a_cols))

    df['question_total_with_alignment'] = df['question_total_score']
    df['answer_total_with_alignment'] = df['answer_total_score']
    df['total_with_alignment'] = df['total_score']
    
    return df

In [ ]:
def calculate_agreement_reliability(expert_dfs_dict, criteria_cols, min_value, max_value):
    import scipy.stats as ss
    results = []
    expert_keys = sorted(expert_dfs_dict.keys())
    if len(expert_keys) < 2:
        print('Need at least 2 experts for agreement analysis')
        return pd.DataFrame()

    allowed = set(range(min_value, max_value + 1))

    def _fleiss_kappa_from_ratings_array(ratings_arr: np.ndarray):
        fleiss_table_local = np.zeros((len(ratings_arr), max_value - min_value + 1), dtype=int)
        for i, item_ratings in enumerate(ratings_arr):
            for r in item_ratings:
                fleiss_table_local[i, r - min_value] += 1
        return float(fleiss_kappa(fleiss_table_local))

    def _fleiss_kappa_permutation_pvalue(ratings_arr: np.ndarray, n_permutations: int = 2000, random_state: int = 42):
        observed = _fleiss_kappa_from_ratings_array(ratings_arr)
        rng = np.random.default_rng(random_state)
        permuted = np.empty_like(ratings_arr)
        extreme = 0
        for _ in range(n_permutations):
            for col in range(ratings_arr.shape[1]):
                permuted[:, col] = rng.permutation(ratings_arr[:, col])
            stat = _fleiss_kappa_from_ratings_array(permuted)
            if abs(stat) >= abs(observed):
                extreme += 1
        return (extreme + 1) / (n_permutations + 1)

    def _unique_valid_rating(series: pd.Series):
        s = pd.to_numeric(series, errors='coerce').dropna()
        s = s[s.isin(allowed)]
        uniques = pd.unique(s)
        if len(uniques) == 1:
            return float(uniques[0])
        return np.nan

    collapsed = {}
    base_common_ids = None

    for expert_key in expert_keys:
        df = expert_dfs_dict[expert_key].copy()
        if 'sample_id' not in df.columns:
            print(f'Skip {expert_key}: missing sample_id column')
            continue

        df['sample_id'] = normalize_sample_id(df['sample_id'])
        keep_cols = ['sample_id'] + [c for c in criteria_cols if c in df.columns]
        df = df[keep_cols].copy()

        for criterion in keep_cols:
            if criterion != 'sample_id':
                df[criterion] = pd.to_numeric(df[criterion], errors='coerce')

        collapsed_df = df.groupby('sample_id', dropna=True).agg({
            c: _unique_valid_rating for c in keep_cols if c != 'sample_id'
        })
        collapsed[expert_key] = collapsed_df

        ids = set(collapsed_df.index.dropna().astype(str))
        base_common_ids = ids if base_common_ids is None else (base_common_ids & ids)

    if not collapsed or not base_common_ids:
        print('No common sample_ids found across experts')
        return pd.DataFrame()

    base_common_ids = sorted(base_common_ids)
    print(f'Analyzing {len(collapsed)} experts with {len(base_common_ids)} common samples')

    for criterion in criteria_cols:
        if any(criterion not in collapsed[k].columns for k in collapsed.keys()):
            continue

        valid_ids = set(base_common_ids)
        for expert_key in collapsed.keys():
            s = collapsed[expert_key][criterion]
            valid_ids &= set(s[s.notna()].index.astype(str))

        valid_ids = sorted(valid_ids)
        if len(valid_ids) < 2:
            continue

        ratings = []
        for sample_id in valid_ids:
            item = []
            ok = True
            for expert_key in collapsed.keys():
                val = collapsed[expert_key].loc[sample_id, criterion]
                if pd.isna(val) or (val not in allowed):
                    ok = False
                    break
                item.append(int(val))
            if ok:
                ratings.append(item)

        if len(ratings) < 2:
            continue

        ratings_array = np.asarray(ratings, dtype=int)
        
        # 1. Fleiss' Kappa calculation (+ permutation p-value)
        kappa = _fleiss_kappa_from_ratings_array(ratings_array)
        fleiss_k_pval = _fleiss_kappa_permutation_pvalue(ratings_array)
        level = (
            'Poor' if kappa < 0 else
            'Slight' if kappa < 0.2 else
            'Fair' if kappa < 0.4 else
            'Moderate' if kappa < 0.6 else
            'Substantial' if kappa < 0.8 else
            'Almost Perfect'
        )
        
        # 2. Kendall's W calculation
        n_items, m_raters = ratings_array.shape
        ranks = np.zeros_like(ratings_array, dtype=float)
        tie_corr = 0
        for j in range(m_raters):
            ranks[:, j] = ss.rankdata(ratings_array[:, j])
            _, counts = np.unique(ratings_array[:, j], return_counts=True)
            t = counts[counts > 1]
            tie_corr += np.sum(t**3 - t)
            
        R_i = np.sum(ranks, axis=1)
        S = np.sum((R_i - np.mean(R_i))**2)
        denom = (m_raters**2 * (n_items**3 - n_items)) - (m_raters * tie_corr)
        kendalls_w = (12 * S) / denom if denom != 0 else np.nan
        kendalls_chi2 = (m_raters * (n_items - 1) * kendalls_w) if pd.notna(kendalls_w) else np.nan
        kendalls_pval = ss.chi2.sf(kendalls_chi2, df=n_items - 1) if pd.notna(kendalls_chi2) else np.nan

        if criterion in EXP2_QUESTION_COLS or criterion == 'q_bloom_rating':
            rating_domain = 'Question'
        elif criterion in EXP2_ANSWER_COLS or criterion == 'a_bloom_rating':
            rating_domain = 'Answer'
        else:
            rating_domain = 'Other'

        results.append({
            'Criterion': criterion,
            'Fleiss_K': round(float(kappa), 3),
            'Fleiss_K_pval': round(float(fleiss_k_pval), 4),
            'Agreement': level,
            'Kendalls_W': round(float(kendalls_w), 3),
            'Kendalls_W_pval': round(float(kendalls_pval), 4) if pd.notna(kendalls_pval) else np.nan,
            'N_Items': int(len(ratings)),
            'N_Raters': int(len(collapsed)),
            'Mean': round(float(ratings_array.mean()), 2),
            'Std': round(float(ratings_array.std()), 2),
            'Scale': f'{min_value}-{max_value}',
            'Domain': rating_domain,
        })

    return pd.DataFrame(results)


def calculate_icc_3_1(expert_dfs_dict, criteria_cols, min_value, max_value):
    results = []
    expert_keys = sorted(expert_dfs_dict.keys())

    if len(expert_keys) < 2:
        print('Need at least 2 experts for ICC analysis')
        return pd.DataFrame()

    selected_expert_keys = expert_keys[:3]
    if len(selected_expert_keys) < 3:
        print('ICC(3,1) requires exactly 3 selected raters')
        return pd.DataFrame()

    allowed = set(range(min_value, max_value + 1))

    def _unique_valid_rating(series: pd.Series):
        s = pd.to_numeric(series, errors='coerce').dropna()
        s = s[s.isin(allowed)]
        uniques = pd.unique(s)
        if len(uniques) == 1:
            return float(uniques[0])
        return np.nan

    collapsed = {}
    base_common_ids = None

    for expert_key in selected_expert_keys:
        df = expert_dfs_dict[expert_key].copy()
        if 'sample_id' not in df.columns:
            continue

        df['sample_id'] = normalize_sample_id(df['sample_id'])
        keep_cols = ['sample_id'] + [c for c in criteria_cols if c in df.columns]
        df = df[keep_cols].copy()

        for criterion in keep_cols:
            if criterion != 'sample_id':
                df[criterion] = pd.to_numeric(df[criterion], errors='coerce')
                df.loc[(df[criterion] < min_value) | (df[criterion] > max_value), criterion] = np.nan

        collapsed_df = df.groupby('sample_id', dropna=True).agg({
            c: _unique_valid_rating for c in keep_cols if c != 'sample_id'
        })
        collapsed[expert_key] = collapsed_df

        ids = set(collapsed_df.index.dropna().astype(str))
        base_common_ids = ids if base_common_ids is None else (base_common_ids & ids)

    if len(collapsed) < 3 or not base_common_ids:
        print('No sufficient common sample_ids found for ICC across 3 selected experts')
        return pd.DataFrame()

    for criterion in criteria_cols:
        if any(criterion not in collapsed[k].columns for k in selected_expert_keys):
            continue

        valid_ids = set(base_common_ids)
        for expert_key in selected_expert_keys:
            s = collapsed[expert_key][criterion]
            valid_ids &= set(s[s.notna()].index.astype(str))

        valid_ids = sorted(valid_ids)
        if len(valid_ids) < 2:
            continue

        long_rows = []
        for sample_id in valid_ids:
            for expert_key in selected_expert_keys:
                val = collapsed[expert_key].loc[sample_id, criterion]
                if pd.isna(val) or (val not in allowed):
                    continue
                long_rows.append({
                    'targets': sample_id,
                    'raters': expert_key,
                    'ratings': float(val),
                })

        icc_input = pd.DataFrame(long_rows)
        if icc_input.empty:
            continue

        per_target_counts = icc_input.groupby('targets')['raters'].nunique()
        complete_targets = per_target_counts[per_target_counts == len(selected_expert_keys)].index
        icc_input = icc_input[icc_input['targets'].isin(complete_targets)].copy()

        if icc_input['targets'].nunique() < 2:
            continue

        icc_table = pg.intraclass_corr(
            data=icc_input,
            targets='targets',
            raters='raters',
            ratings='ratings',
            nan_policy='omit',
        )
        icc_row = icc_table.loc[icc_table['Type'] == 'ICC3']
        if icc_row.empty:
            continue

        icc_row = icc_row.iloc[0]
        ci95 = icc_row.get('CI95', np.nan)
        if isinstance(ci95, (list, tuple, np.ndarray, pd.Series)):
            ci95_vals = pd.Series(ci95).dropna().tolist()
            ci95_str = str(ci95_vals) if len(ci95_vals) > 0 else np.nan
        elif pd.isna(ci95):
            ci95_str = np.nan
        else:
            ci95_str = str(ci95)

        if criterion in EXP2_QUESTION_COLS or criterion == 'q_bloom_rating':
            rating_domain = 'Question'
        elif criterion in EXP2_ANSWER_COLS or criterion == 'a_bloom_rating':
            rating_domain = 'Answer'
        else:
            rating_domain = 'Other'

        results.append({
            'Criterion': criterion,
            'Display_Label': DISPLAY_LABELS.get(criterion, criterion),
            'ICC3_1': round(float(icc_row['ICC']), 3),
            'ICC3_1_F': round(float(icc_row['F']), 3),
            'ICC3_1_df1': int(icc_row['df1']),
            'ICC3_1_df2': int(icc_row['df2']),
            'ICC3_1_pval': round(float(icc_row['pval']), 4),
            'ICC3_1_CI95': ci95_str,
            'N_Items': int(icc_input['targets'].nunique()),
            'N_Raters': int(len(selected_expert_keys)),
            'Scale': f'{min_value}-{max_value}',
            'Rating_Domain': rating_domain,
        })

    return pd.DataFrame(results)


print('Scoring and agreement helpers loaded successfully')

## Data Loading and Configuration

In [ ]:
print(f'Loading unified expert data (Experts {EXPERT_IDS[0]}-{EXPERT_IDS[-1]})')
exp2_df, experts_data = load_all_experts_data()
agreement_available = len(experts_data) >= 2
analysis_suffix = ANALYSIS_SUFFIX

exp2_df = clean_numeric_data(exp2_df, EXP2_QUESTION_COLS + EXP2_ANSWER_COLS, LIKERT_MIN, LIKERT_MAX)
exp2_df = clean_numeric_data(exp2_df, EXP2_BLOOM_COLS, BLOOM_MIN, BLOOM_MAX, use_semicolon_max=True)
exp2_df = calculate_exp2_scores(exp2_df)
criteria = EXP2_NUMERIC_COLS + EXP2_SCORE_COLS

filled_non_bloom = exp2_df[EXP2_QUESTION_COLS + EXP2_ANSWER_COLS].notna().sum().sum()
total_non_bloom = len(exp2_df) * len(EXP2_QUESTION_COLS + EXP2_ANSWER_COLS)
completion_pct = (100 * filled_non_bloom / total_non_bloom) if total_non_bloom else 0

print('\nData loaded successfully!')
print(f'Experiment 2: {len(exp2_df)} rows, {completion_pct:.1f}% non-Bloom rating completion')
print(f'Experts loaded: {list(experts_data.keys())}')

if 'llm' in exp2_df.columns:
    print('\nDistribution:')
    print(f"  LLMs: {list(exp2_df['llm'].dropna().unique())}")
if 'question_type' in exp2_df.columns:
    print(f"  Question Types: {list(exp2_df['question_type'].dropna().unique())}")
if 'bloom_idx' in exp2_df.columns:
    print(f"  Bloom Levels: {sorted(exp2_df['bloom_idx'].dropna().unique())}")

# Experiment 2: Learning Objective and Bloom Alignment Analysis

Analysis of LLM question generation quality with explicit Bloom-level alignment criteria.

In [ ]:
print('EXPERIMENT 2 - DESCRIPTIVE STATISTICS')
print('=' * 60)

criteria = EXP2_NUMERIC_COLS + EXP2_SCORE_COLS
exp2_stats = exp2_df[criteria].describe().round(2)
tables[f'exp2_overall_stats_{analysis_suffix}'] = exp2_stats
print('\nOverall Statistics:')
display(exp2_stats)

exp2_llm_stats = exp2_df.groupby('llm')[criteria].agg(['mean', 'std', 'median', 'count']).round(2)
tables[f'exp2_llm_stats_{analysis_suffix}'] = exp2_llm_stats
print('\nStatistics by LLM:')
display(exp2_llm_stats)

ranking_cols = ['question_total_score', 'answer_total_score', 'total_score']
exp2_llm_means = exp2_df.groupby('llm')[ranking_cols].mean().round(2)
exp2_llm_means['combined_mean'] = exp2_llm_means.mean(axis=1)
exp2_llm_overall = exp2_llm_means.sort_values('combined_mean', ascending=False)
tables[f'exp2_llm_ranking_{analysis_suffix}'] = exp2_llm_overall
print('\nOverall LLM Ranking:')
display(exp2_llm_overall)

exp2_question_type_stats = exp2_df.groupby('question_type')[criteria].agg(['mean', 'std', 'median', 'count']).round(2)
tables[f'exp2_question_type_stats_{analysis_suffix}'] = exp2_question_type_stats
print('\nStatistics by Question Type:')
display(exp2_question_type_stats)

In [ ]:
question_plot_metrics = EXP2_QUESTION_COLS + ['q_bloom_alignment', 'question_total_score']

fig, axes = plt.subplots(3, 2, figsize=(14, 14))
axes = axes.flatten()
plots[f'exp2_question_bundle_question_type_{analysis_suffix}'] = fig

for i, criterion in enumerate(question_plot_metrics):
    scale_range = SCORE_RANGES.get(criterion, LIKERT_RANGE)
    create_seaborn_boxplot(
        exp2_df,
        'question_type',
        criterion,
        axes[i],
        DISPLAY_LABELS.get(criterion, criterion.title()),
        DISPLAY_LABELS.get(criterion, criterion.title()),
        'Question Type',
        scale_range=scale_range,
    )

plt.suptitle('Experiment 2: Question-based Criteria', fontsize=16, fontweight='bold', y=0.98)
plt.tight_layout()
plt.subplots_adjust(top=0.94)
plt.show()

answer_plot_metrics = EXP2_ANSWER_COLS + ['a_bloom_alignment', 'answer_total_score']
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()
plots[f'exp2_answer_bundle_question_type_{analysis_suffix}'] = fig

for i, criterion in enumerate(answer_plot_metrics):
    scale_range = SCORE_RANGES.get(criterion, LIKERT_RANGE)
    create_seaborn_boxplot(
        exp2_df,
        'question_type',
        criterion,
        axes[i],
        DISPLAY_LABELS.get(criterion, criterion.title()),
        DISPLAY_LABELS.get(criterion, criterion.title()),
        'Question Type',
        scale_range=scale_range,
    )

plt.suptitle('Experiment 2: Answer-based Criteria', fontsize=16, fontweight='bold', y=0.98)
plt.tight_layout()
plt.subplots_adjust(top=0.92)
plt.show()

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(14, 14))
axes = axes.flatten()
plots[f'exp2_question_bundle_llm_{analysis_suffix}'] = fig

for i, criterion in enumerate(question_plot_metrics):
    scale_range = SCORE_RANGES.get(criterion, LIKERT_RANGE)
    create_seaborn_boxplot(
        exp2_df,
        'llm',
        criterion,
        axes[i],
        DISPLAY_LABELS.get(criterion, criterion.title()),
        DISPLAY_LABELS.get(criterion, criterion.title()),
        'LLM',
        scale_range=scale_range,
    )

plt.suptitle('Experiment 2: Question-based Criteria by LLM', fontsize=16, fontweight='bold', y=0.98)
plt.tight_layout()
plt.subplots_adjust(top=0.94)
plt.show()

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()
plots[f'exp2_answer_bundle_llm_{analysis_suffix}'] = fig

for i, criterion in enumerate(answer_plot_metrics):
    scale_range = SCORE_RANGES.get(criterion, LIKERT_RANGE)
    create_seaborn_boxplot(
        exp2_df,
        'llm',
        criterion,
        axes[i],
        DISPLAY_LABELS.get(criterion, criterion.title()),
        DISPLAY_LABELS.get(criterion, criterion.title()),
        'LLM',
        scale_range=scale_range,
    )

plt.suptitle('Experiment 2: Answer-based Criteria by LLM', fontsize=16, fontweight='bold', y=0.98)
plt.tight_layout()
plt.subplots_adjust(top=0.92)
plt.show()

In [ ]:
exp2_df_plot = exp2_df.copy()
exp2_df_plot['bloom_idx'] = pd.to_numeric(exp2_df_plot['bloom_idx'], errors='coerce')
exp2_df_plot = exp2_df_plot[exp2_df_plot['bloom_idx'].between(1, 6, inclusive='both')].copy()
LABEL_MAPPING.update({str(i): f'{i}' for i in range(1, 7)})

fig, axes = plt.subplots(3, 2, figsize=(14, 14))
axes = axes.flatten()
plots[f'exp2_question_bundle_bloom_{analysis_suffix}'] = fig

for i, criterion in enumerate(question_plot_metrics):
    scale_range = SCORE_RANGES.get(criterion, LIKERT_RANGE)
    create_seaborn_boxplot(
        exp2_df_plot,
        'bloom_idx',
        criterion,
        axes[i],
        DISPLAY_LABELS.get(criterion, criterion.title()),
        DISPLAY_LABELS.get(criterion, criterion.title()),
        'Bloom Level',
        scale_range=scale_range,
    )

plt.suptitle('Experiment 2: Question-based Criteria by Bloom Level', fontsize=16, fontweight='bold', y=0.98)
plt.tight_layout()
plt.subplots_adjust(top=0.94)
plt.show()

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()
plots[f'exp2_answer_bundle_bloom_{analysis_suffix}'] = fig

for i, criterion in enumerate(answer_plot_metrics):
    scale_range = SCORE_RANGES.get(criterion, LIKERT_RANGE)
    create_seaborn_boxplot(
        exp2_df_plot,
        'bloom_idx',
        criterion,
        axes[i],
        DISPLAY_LABELS.get(criterion, criterion.title()),
        DISPLAY_LABELS.get(criterion, criterion.title()),
        'Bloom Level',
        scale_range=scale_range,
    )

plt.suptitle('Experiment 2: Answer-based Criteria by Bloom Level', fontsize=16, fontweight='bold', y=0.98)
plt.tight_layout()
plt.subplots_adjust(top=0.92)
plt.show()

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(10, 7))
plots[f'exp2_question_total_heatmap_{analysis_suffix}'] = fig
create_heatmap(
    exp2_df,
    'llm',
    'question_type',
    'question_total_with_alignment',
    ax,
    'Question Total: LLM vs Question Type\nValues: Mean (Std) | Scale: 5-25 points',
    'Question Total',
    scale_range=SCORE_RANGES['question_total_with_alignment'],
)
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(1, 1, figsize=(10, 7))
plots[f'exp2_answer_total_heatmap_{analysis_suffix}'] = fig
create_heatmap(
    exp2_df,
    'llm',
    'question_type',
    'answer_total_with_alignment',
    ax,
    'Answer Total: LLM vs Question Type\nValues: Mean (Std) | Scale: 3-15 points',
    'Answer Total',
    scale_range=SCORE_RANGES['answer_total_with_alignment'],
)
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(1, 1, figsize=(10, 7))
plots[f'exp2_total_score_heatmap_{analysis_suffix}'] = fig
create_heatmap(
    exp2_df,
    'llm',
    'question_type',
    'total_with_alignment',
    ax,
    'Total Score: LLM vs Question Type\nValues: Mean (Std) | Scale: 8-40 points',
    'Overall Total',
    scale_range=SCORE_RANGES['total_with_alignment'],
)
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
plots[f'exp2_bloom_alignment_heatmaps_{analysis_suffix}'] = fig
create_heatmap(
    exp2_df,
    'llm',
    'question_type',
    'q_bloom_alignment',
    axes[0],
    'Question Bloom Alignment: LLM vs Question Type\nValues: Mean (Std) | Scale: 1-5',
    'Question Bloom Alignment',
    scale_range=SCORE_RANGES['q_bloom_alignment'],
 )
create_heatmap(
    exp2_df,
    'llm',
    'question_type',
    'a_bloom_alignment',
    axes[1],
    'Answer Bloom Alignment: LLM vs Question Type\nValues: Mean (Std) | Scale: 1-5',
    'Answer Bloom Alignment',
    scale_range=SCORE_RANGES['a_bloom_alignment'],
 )
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
plots[f'exp2_bloom_confusion_matrices_{analysis_suffix}'] = fig

valid_q = exp2_df.dropna(subset=['bloom_idx', 'q_bloom_rating']).copy()
valid_q['bloom_idx'] = valid_q['bloom_idx'].astype(int)
valid_q['q_bloom_rating'] = valid_q['q_bloom_rating'].astype(int)

q_cm_counts = pd.crosstab(valid_q['q_bloom_rating'], valid_q['bloom_idx'])
q_cm_pct = pd.crosstab(valid_q['q_bloom_rating'], valid_q['bloom_idx'], normalize='columns') * 100

for i in range(1, 7):
    if i not in q_cm_counts.index: 
        q_cm_counts.loc[i] = 0
        q_cm_pct.loc[i] = 0
    if i not in q_cm_counts.columns: 
        q_cm_counts[i] = 0
        q_cm_pct[i] = 0
q_cm_counts = q_cm_counts.sort_index().sort_index(axis=1)
q_cm_pct = q_cm_pct.sort_index().sort_index(axis=1)

q_annot = np.empty_like(q_cm_counts, dtype=object)
for i in range(q_cm_counts.shape[0]):
    for j in range(q_cm_counts.shape[1]):
        count = q_cm_counts.iloc[i, j]
        pct = q_cm_pct.iloc[i, j]
        q_annot[i, j] = f"{count}\n({pct:.1f}%)" if count > 0 else "0\n(0.0%)"

sns.heatmap(q_cm_pct, annot=q_annot, fmt='', cmap='Blues', ax=axes[0], cbar_kws={'label': 'Percentage %'})
axes[0].set_title('Question: Target vs Actual Bloom Level\nValues: Count (Percentage)', fontsize=14, fontweight='bold', pad=20)
axes[0].set_xlabel('Target Bloom Level (bloom_idx)', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Actual Question Bloom Level (q_bloom_rating)', fontsize=12, fontweight='bold')
axes[0].invert_yaxis()

valid_a = exp2_df.dropna(subset=['q_bloom_rating', 'a_bloom_rating']).copy()
valid_a['q_bloom_rating'] = valid_a['q_bloom_rating'].astype(int)
valid_a['a_bloom_rating'] = valid_a['a_bloom_rating'].astype(int)

a_cm_counts = pd.crosstab(valid_a['a_bloom_rating'], valid_a['q_bloom_rating'])
a_cm_pct = pd.crosstab(valid_a['a_bloom_rating'], valid_a['q_bloom_rating'], normalize='columns') * 100

for i in range(1, 7):
    if i not in a_cm_counts.index: 
        a_cm_counts.loc[i] = 0
        a_cm_pct.loc[i] = 0
    if i not in a_cm_counts.columns: 
        a_cm_counts[i] = 0
        a_cm_pct[i] = 0
a_cm_counts = a_cm_counts.sort_index().sort_index(axis=1)
a_cm_pct = a_cm_pct.sort_index().sort_index(axis=1)

a_annot = np.empty_like(a_cm_counts, dtype=object)
for i in range(a_cm_counts.shape[0]):
    for j in range(a_cm_counts.shape[1]):
        count = a_cm_counts.iloc[i, j]
        pct = a_cm_pct.iloc[i, j]
        a_annot[i, j] = f"{count}\n({pct:.1f}%)" if count > 0 else "0\n(0.0%)"

sns.heatmap(a_cm_pct, annot=a_annot, fmt='', cmap='Oranges', ax=axes[1], cbar_kws={'label': 'Percentage %'})
axes[1].set_title('Answer: Question Actual vs Answer Actual Bloom Level\nValues: Count (Percentage)', fontsize=14, fontweight='bold', pad=20)
axes[1].set_xlabel('Actual Question Bloom Level (q_bloom_rating)', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Actual Answer Bloom Level (a_bloom_rating)', fontsize=12, fontweight='bold')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

# Inter-Rater Agreement Analysis

Agreement is computed across the unified set of available experts. Core question/answer criteria and Bloom alignment scores are analyzed on the 1-5 scale.

In [ ]:
print("INTER-RATER AGREEMENT ANALYSIS (Fleiss' Kappa, Kendall's W, ICC(3,1))")
print('=' * 60)
print(f'Analyzing agreement across {len(EXPERT_IDS)} experts: {EXPERT_IDS}')

expert_dfs_dict = {}
for expert_num in EXPERT_IDS:
    expert_key = f'expert_{expert_num}'
    if expert_key not in experts_data:
        continue
    expert_df = experts_data[expert_key].copy()
    expert_df = clean_numeric_data(expert_df, EXP2_QUESTION_COLS + EXP2_ANSWER_COLS, LIKERT_MIN, LIKERT_MAX)
    expert_df = clean_numeric_data(expert_df, EXP2_BLOOM_COLS, BLOOM_MIN, BLOOM_MAX, use_semicolon_max=True)
    expert_df = calculate_exp2_scores(expert_df)
    expert_dfs_dict[expert_key] = expert_df

agreement_non_bloom = calculate_agreement_reliability(
    expert_dfs_dict,
    EXP2_QUESTION_COLS + EXP2_ANSWER_COLS,
    LIKERT_MIN,
    LIKERT_MAX,
 )
agreement_bloom = calculate_agreement_reliability(
    expert_dfs_dict,
    ['q_bloom_rating', 'a_bloom_rating'],
    BLOOM_MIN,
    BLOOM_MAX,
)
icc_non_bloom = calculate_icc_3_1(
    expert_dfs_dict,
    EXP2_QUESTION_COLS + EXP2_ANSWER_COLS,
    LIKERT_MIN,
    LIKERT_MAX,
)
icc_bloom = calculate_icc_3_1(
    expert_dfs_dict,
    ['q_bloom_rating', 'a_bloom_rating'],
    BLOOM_MIN,
    BLOOM_MAX,
)

agreement_exp2 = pd.concat([agreement_non_bloom, agreement_bloom], ignore_index=True)
icc_exp2 = pd.concat([icc_non_bloom, icc_bloom], ignore_index=True)
if not icc_exp2.empty:
    agreement_exp2 = agreement_exp2.merge(
        icc_exp2[['Criterion', 'ICC3_1', 'ICC3_1_CI95', 'ICC3_1_pval']],
        on='Criterion',
        how='left',
    )
agreement_exp2 = agreement_exp2.sort_values(
    'Criterion',
    key=lambda s: s.str.startswith('a_').astype(int),
    kind='stable'
).reset_index(drop=True)
core_cols = ['Criterion', 'Fleiss_K', 'Fleiss_K_pval', 'Agreement', 'Kendalls_W', 'Kendalls_W_pval', 'ICC3_1', 'ICC3_1_CI95', 'ICC3_1_pval']
ordered_cols = [c for c in core_cols if c in agreement_exp2.columns]
remaining_cols = [c for c in agreement_exp2.columns if c not in ordered_cols]
agreement_exp2 = agreement_exp2[ordered_cols + remaining_cols]

if not agreement_exp2.empty:
    tables[f'agreement_exp2_{analysis_suffix}'] = agreement_exp2
    display(agreement_exp2.round(3))

    valid_kappas = agreement_exp2['Fleiss_K'].dropna()
    valid_ws = agreement_exp2['Kendalls_W'].dropna()
    valid_icc = agreement_exp2['ICC3_1'].dropna() if 'ICC3_1' in agreement_exp2.columns else pd.Series(dtype=float)
    if len(valid_kappas) > 0:
        avg_kappa = valid_kappas.mean()
        avg_w = valid_ws.mean()
        level = (
            'Poor' if avg_kappa < 0 else
            'Slight' if avg_kappa < 0.2 else
            'Fair' if avg_kappa < 0.4 else
            'Moderate' if avg_kappa < 0.6 else
            'Substantial' if avg_kappa < 0.8 else
            'Almost Perfect'
        )
        print(f"\nAverage Fleiss' Kappa: {avg_kappa:.3f} ({level})")
        print(f"Average Kendall's W: {avg_w:.3f}")
    if len(valid_icc) > 0:
        print(f"Average ICC(3,1): {valid_icc.mean():.3f}")

print('\n' + '=' * 60)
print('EXPERT COMPARISON - INDIVIDUAL STATISTICS')
print('=' * 60)

expert_means = {}
available_expert_keys = [k for k in [f'expert_{n}' for n in EXPERT_IDS] if k in experts_data]

for expert_key in available_expert_keys:
    cleaned_expert_df = clean_numeric_data(experts_data[expert_key], EXP2_QUESTION_COLS + EXP2_ANSWER_COLS, LIKERT_MIN, LIKERT_MAX)
    cleaned_expert_df = clean_numeric_data(cleaned_expert_df, EXP2_BLOOM_COLS, BLOOM_MIN, BLOOM_MAX, use_semicolon_max=True)
    cleaned_expert_df = calculate_exp2_scores(cleaned_expert_df)
    expert_means[expert_key] = cleaned_expert_df[EXP2_NUMERIC_COLS + EXP2_SCORE_COLS].mean()

comparison_df = pd.DataFrame(expert_means).T
comparison_df = comparison_df.round(3)
tables[f'expert_comparison_exp2_{analysis_suffix}'] = comparison_df

print('\nMean Ratings by Expert (Experiment 2):')
display(comparison_df)

In [ ]:
if not agreement_exp2.empty:
    summary_cols = ['Fleiss_K', 'Kendalls_W', 'Mean', 'Std']
    if 'ICC3_1' in agreement_exp2.columns:
        summary_cols.append('ICC3_1')
    agreement_summary = agreement_exp2.groupby('Domain')[summary_cols].mean().round(3)
    tables[f'agreement_summary_exp2_{analysis_suffix}'] = agreement_summary
    print('Average agreement by domain:')
    display(agreement_summary)
else:
    print('No agreement summary available.')

## Data Export

Save all tables and plots for further analysis and reporting.

In [ ]:
for table_name, table_df in tables.items():
    if isinstance(table_df, pd.DataFrame):
        safe_name = table_name.replace(' ', '_').replace('(', '').replace(')', '').replace("'", '').lower()
        csv_path = os.path.join(output_tables_path, f'{safe_name}.csv')
        table_df.to_csv(csv_path)
        print(f'Saved table: {csv_path}')

for plot_name, fig in plots.items():
    safe_name = plot_name.replace(' ', '_').replace('(', '').replace(')', '').replace("'", '').lower()
    png_path = os.path.join(output_plots_path, f'{safe_name}.png')
    fig.savefig(png_path, dpi=300, bbox_inches='tight')
    print(f'Saved plot: {png_path}')

print(f'\nAnalysis complete. Results saved to: {output_base_path}')